## Install the required libraries

Run the following if the packages are not already installed.

In [ ]:
# Uncomment if required:
# %pip install nltk spacy textblob pandas numpy
# !python -m spacy download en_core_web_sm

In [1]:
import math
import numpy as np
import pandas as pd
import nltk
import spacy

from collections import Counter
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from nltk.util import ngrams
from nltk import pos_tag
from textblob import TextBlob

print("NLTK:", nltk.__version__)
print("spaCy:", spacy.__version__)

NLTK: 3.10.0
spaCy: 3.8.16


## Download NLTK resources

These resources support tokenization, WordNet lemmatization, POS tagging and stopwords.

In [2]:
resources = [
    "punkt", "punkt_tab", "wordnet", "omw-1.4",
    "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
    "stopwords"
]

for resource in resources:
    nltk.download(resource, quiet=True)

print("NLTK resources downloaded.")

NLTK resources downloaded.


## Load the spaCy English model

The `en_core_web_sm` model provides tokenization, POS tagging, lemmatization and NER.

If it is unavailable, run:

```python
!python -m spacy download en_core_web_sm
```

and restart the kernel.

In [3]:
try:
    nlp = spacy.load("en_core_web_sm")
    print("spaCy model loaded.")
except OSError:
    nlp = None
    print("Model not found. Run: !python -m spacy download en_core_web_sm")

spaCy model loaded.


# Sample text


In [4]:
text = """Riya is working at Infosys in Bengaluru.
She is analyzing customer feedback and building an AI model.
The team completed the project in September 2026."""

print(text)

Riya is working at Infosys in Bengaluru.
She is analyzing customer feedback and building an AI model.
The team completed the project in September 2026.


# Tokenization

**Tokenization** breaks text into smaller units called tokens.

- Sentence tokenization → sentences
- Word tokenization → words/punctuation

Example:

`"AI is useful."` → `["AI", "is", "useful", "."]`

In [5]:
sentences = sent_tokenize(text)

print("Sentence tokens:")
for i, sentence in enumerate(sentences, 1):
    print(i, "->", sentence)

Sentence tokens:
1 -> Riya is working at Infosys in Bengaluru.
2 -> She is analyzing customer feedback and building an AI model.
3 -> The team completed the project in September 2026.


In [6]:
tokens = word_tokenize(text)

print("Word tokens:")
print(tokens)

Word tokens:
['Riya', 'is', 'working', 'at', 'Infosys', 'in', 'Bengaluru', '.', 'She', 'is', 'analyzing', 'customer', 'feedback', 'and', 'building', 'an', 'AI', 'model', '.', 'The', 'team', 'completed', 'the', 'project', 'in', 'September', '2026', '.']


In [7]:
if nlp:
    doc = nlp(text)
    print("spaCy tokens:")
    print([token.text for token in doc])
else:
    doc = None

spaCy tokens:
['Riya', 'is', 'working', 'at', 'Infosys', 'in', 'Bengaluru', '.', '\n', 'She', 'is', 'analyzing', 'customer', 'feedback', 'and', 'building', 'an', 'AI', 'model', '.', '\n', 'The', 'team', 'completed', 'the', 'project', 'in', 'September', '2026', '.']


### Key idea

Tokenization is normally one of the first NLP steps:

**Raw text → tokens → linguistic analysis → features → model**

# Stemming

**Stemming** reduces words to a root-like form by removing or changing endings.

For example:

- `playing` → `play`
- `studies` → `studi`
- `connected` → `connect`

A stem is not necessarily a valid dictionary word.

In [8]:
words = [
    "play", "playing", "played",
    "study", "studies", "studying",
    "connection", "connected", "connecting"
]

porter = PorterStemmer()
snowball = SnowballStemmer("english")

stemming_df = pd.DataFrame({
    "Word": words,
    "Porter": [porter.stem(w) for w in words],
    "Snowball": [snowball.stem(w) for w in words]
})

stemming_df

,Word,Porter,Snowball
0,play,play,play
1,playing,play,play
2,played,play,play
3,study,studi,studi
4,studies,studi,studi
5,studying,studi,studi
6,connection,connect,connect
7,connected,connect,connect
8,connecting,connect,connect


### Stemming vs Lemmatization

**Stemming:** fast and simple, but may produce non-words.

**Lemmatization:** uses linguistic information and generally produces a valid base form.

# Lemmatization with NLTK

Lemmatization finds the dictionary/base form of a word.

Examples:

- `cars` → `car`
- `studies` → `study`
- `running` → `run` when treated as a verb

NLTK's `WordNetLemmatizer` works better when the appropriate POS is supplied.

In [9]:
lemmatizer = WordNetLemmatizer()

word_pos = [
    ("cars", "n"),
    ("running", "v"),
    ("studies", "v"),
    ("played", "v"),
    ("better", "a")
]

rows = [
    (word, pos, lemmatizer.lemmatize(word, pos=pos))
    for word, pos in word_pos
]

pd.DataFrame(rows, columns=["Word", "WordNet POS", "Lemma"])

,Word,WordNet POS,Lemma
0,cars,n,car
1,running,v,run
2,studies,v,study
3,played,v,play
4,better,a,good


# Part-of-Speech (POS) Tagging

POS tagging identifies the grammatical role of a word.

Common categories:

- NOUN – noun
- VERB – verb
- ADJ – adjective
- ADV – adverb
- PRON – pronoun
- DET – determiner
- PROPN – proper noun
- ADP – preposition

Different NLP libraries use different tagsets.

## POS tagging with NLTK

In [11]:
nltk_tags = pos_tag(word_tokenize(text))

for word, tag in nltk_tags:
    print(f"{word:15} {tag}")

Riya            NNP
is              VBZ
working         VBG
at              IN
Infosys         NNP
in              IN
Bengaluru       NNP
.               .
She             PRP
is              VBZ
analyzing       VBG
customer        NN
feedback        NN
and             CC
building        NN
an              DT
AI              NNP
model           NN
.               .
The             DT
team            NN
completed       VBD
the             DT
project         NN
in              IN
September       NNP
2026            CD
.               .


## POS tagging with TextBlob

In [12]:
blob = TextBlob(text)

textblob_tags = pd.DataFrame(
    blob.tags,
    columns=["Word", "POS"]
)

textblob_tags

,Word,POS
0,Riya,NNP
1,is,VBZ
2,working,VBG
3,at,IN
4,Infosys,NNP
5,in,IN
6,Bengaluru,NNP
7,She,PRP
8,is,VBZ
9,analyzing,VBG


# N-grams

An **N-gram** is a sequence of `N` consecutive tokens.

- Unigram → one token
- Bigram → two tokens
- Trigram → three tokens

Example:

`machine learning model`

Bigrams:

`machine learning`, `learning model`

Trigrams:

`machine learning model`

In [14]:
ngram_text = "natural language processing is useful for machine learning"
ngram_tokens = word_tokenize(ngram_text)

unigrams = list(ngrams(ngram_tokens, 1))
bigrams = list(ngrams(ngram_tokens, 2))
trigrams = list(ngrams(ngram_tokens, 3))

print("Unigrams:", unigrams)
print("\nBigrams:")
for item in bigrams:
    print(" ".join(item))

print("\nTrigrams:")
for item in trigrams:
    print(" ".join(item))

Unigrams: [('natural',), ('language',), ('processing',), ('is',), ('useful',), ('for',), ('machine',), ('learning',)]

Bigrams:
natural language
language processing
processing is
is useful
useful for
for machine
machine learning

Trigrams:
natural language processing
language processing is
processing is useful
is useful for
useful for machine
for machine learning


### Applications of N-grams

N-grams can capture local word-order information and are useful in:

- Search
- Text classification
- Autocomplete
- Spam detection
- Language modeling

# One-Hot Encoding

One-hot encoding represents each vocabulary word as a binary vector.

Vocabulary:

`["ai", "data", "learning", "python"]`

Representation:

```text
ai       → [1, 0, 0, 0]
data     → [0, 1, 0, 0]
learning → [0, 0, 1, 0]
python   → [0, 0, 0, 1]
```

Only one position contains `1`.

In [15]:
vocabulary = ["ai", "data", "learning", "python"]

one_hot = {}

for i, word in enumerate(vocabulary):
    vector = [0] * len(vocabulary)
    vector[i] = 1
    one_hot[word] = vector

one_hot_df = pd.DataFrame.from_dict(
    one_hot,
    orient="index",
    columns=vocabulary
)

one_hot_df

,ai,data,learning,python
ai,1,0,0,0
data,0,1,0,0
learning,0,0,1,0
python,0,0,0,1


### Limitation

One-hot vectors do not encode semantic similarity.

For example, the vectors for `cat` and `dog` are just different binary positions; the representation itself does not tell the model that the two concepts are related.

# Bag of Words (BoW)

Bag of Words represents each document by the number of times each vocabulary word occurs.

It **ignores word order**.

Example:

```text
D1 = Python is easy
D2 = Python is powerful
```

Vocabulary:

`Python, is, easy, powerful`

D1 → `[1, 1, 1, 0]`

D2 → `[1, 1, 0, 1]`

In [16]:
documents = [
    "python is easy",
    "python is powerful",
    "python is easy and powerful"
]

tokenized_documents = [doc.lower().split() for doc in documents]

vocabulary = sorted(
    set(word for doc in tokenized_documents for word in doc)
)

print("Vocabulary:", vocabulary)

Vocabulary: ['and', 'easy', 'is', 'powerful', 'python']


In [17]:
bow_matrix = []

for tokens in tokenized_documents:
    counts = Counter(tokens)
    bow_matrix.append([counts[word] for word in vocabulary])

bow_df = pd.DataFrame(
    bow_matrix,
    columns=vocabulary,
    index=["D1", "D2", "D3"]
)

bow_df

,and,easy,is,powerful,python
D1,0,1,1,0,1
D2,0,0,1,1,1
D3,1,1,1,1,1


### Limitation of BoW

BoW loses word order.

For example:

- `Dog bites man`
- `Man bites dog`

contain the same words but have different meanings.

N-grams can preserve some local word-order information.

# TF-IDF

**TF-IDF = Term Frequency × Inverse Document Frequency**

It gives greater weight to terms that are important in a document but relatively uncommon across the collection.

A common formulation is:

\[
TF(t,d) = \frac{count(t,d)}{total\ terms\ in\ d}
\]

\[
IDF(t) = \log_{10}\left(\frac{N}{DF(t)}\right)
\]

\[
TFIDF(t,d) = TF(t,d) \times IDF(t)
\]

Where:

- `N` = number of documents
- `DF(t)` = number of documents containing term `t`

## Manual TF-IDF example

Consider:

```text
D1 = python data science
D2 = python machine learning
D3 = machine learning data
```

For `data` in D1:

- TF = 1/3
- DF = 2 because `data` appears in D1 and D3
- N = 3
- IDF = log10(3/2)
- TF-IDF = TF × IDF

The exact values can vary between libraries because production implementations often use smoothing and vector normalization.

In [18]:
tfidf_documents = [
    "python data science",
    "python machine learning",
    "machine learning data"
]

tokenized = [doc.lower().split() for doc in tfidf_documents]
vocab = sorted(set(word for doc in tokenized for word in doc))

N = len(tokenized)

df = {
    word: sum(1 for doc in tokenized if word in doc)
    for word in vocab
}

idf = {
    word: math.log10(N / df[word])
    for word in vocab
}

print("Vocabulary:", vocab)
print("DF:", df)
print("IDF:", {word: round(value, 4) for word, value in idf.items()})

Vocabulary: ['data', 'learning', 'machine', 'python', 'science']
DF: {'data': 2, 'learning': 2, 'machine': 2, 'python': 2, 'science': 1}
IDF: {'data': 0.1761, 'learning': 0.1761, 'machine': 0.1761, 'python': 0.1761, 'science': 0.4771}


In [19]:
tfidf_matrix = []

for tokens in tokenized:
    counts = Counter(tokens)
    total_terms = len(tokens)

    row = []
    for word in vocab:
        tf = counts[word] / total_terms
        tfidf = tf * idf[word]
        row.append(tfidf)

    tfidf_matrix.append(row)

tfidf_df = pd.DataFrame(
    tfidf_matrix,
    columns=vocab,
    index=["D1", "D2", "D3"]
)

tfidf_df.round(4)

,data,learning,machine,python,science
D1,0.0587,0.0000,0.0000,0.0587,0.159
D2,0.0000,0.0587,0.0587,0.0587,0.000
D3,0.0587,0.0587,0.0587,0.0000,0.000


### Interpretation

If a word occurs in almost every document, its IDF becomes small.

If a word occurs in only a few documents, its IDF is larger.

Therefore TF-IDF emphasizes **distinctive words**.

Common applications:

- Document classification
- Search
- Document similarity
- Keyword extraction
- Spam detection

# Named Entity Recognition (NER)

**Named Entity Recognition** identifies important real-world entities in text.

Examples:

- PERSON → Riya
- ORG → Infosys
- GPE → Bengaluru
- DATE → September 2026
- MONEY → ₹50,000

spaCy provides a pretrained NER pipeline.

In [20]:
if doc:
    entities = [
        {
            "Entity": ent.text,
            "Label": ent.label_,
            "Meaning": spacy.explain(ent.label_)
        }
        for ent in doc.ents
    ]

    pd.DataFrame(entities)

In [21]:
if doc:
    for ent in doc.ents:
        print(f"{ent.text:20} -> {ent.label_:8} -> {spacy.explain(ent.label_)}")

Riya                 -> PERSON   -> People, including fictional
Infosys              -> ORG      -> Companies, agencies, institutions, etc.
Bengaluru            -> GPE      -> Countries, cities, states
AI                   -> GPE      -> Countries, cities, states
September 2026       -> DATE     -> Absolute or relative dates or periods


## Visualizing NER

spaCy can highlight entities directly in a Jupyter notebook.

In [22]:
from spacy import displacy

if doc:
    displacy.render(doc, style="ent", jupyter=True)

# Complete NLP Pipeline

A simplified NLP workflow can look like:

```text
Raw Text
   ↓
Sentence Tokenization
   ↓
Word Tokenization
   ↓
Cleaning / Normalization
   ↓
POS Tagging / Lemmatization
   ↓
Feature Extraction
   ├── One-Hot Encoding
   ├── Bag of Words
   ├── N-grams
   └── TF-IDF
   ↓
Machine Learning / NLP Model
   ↓
Prediction / Classification / Search
```

NER can be applied when the goal is to extract people, organizations, locations, dates and other entities.

#  Technique Comparison

| Technique | Main purpose |
|---|---|
| Tokenization | Break text into tokens |
| Stemming | Reduce words to a rough root |
| Lemmatization | Find the dictionary/base form |
| POS Tagging | Identify grammatical role |
| N-grams | Capture word sequences |
| One-Hot Encoding | Represent words as binary vectors |
| Bag of Words | Represent documents using word counts |
| TF-IDF | Weight distinctive terms |
| NER | Extract named entities |

### Easy memory aid

**Tokenization → Break**

**Stemming → Chop**

**Lemmatization → Dictionary root**

**POS → Grammar role**

**N-gram → Word sequence**

**One-hot → Binary identity**

**BoW → Count**

**TF-IDF → Importance**

**NER → Real-world entities**

# Points To remember

- **NLTK** is excellent for learning and implementing traditional NLP techniques.
- **spaCy** provides a practical NLP pipeline with POS tagging, lemmatization and NER.
- **TextBlob** provides a beginner-friendly interface for text processing and POS tagging.
- Tokenization prepares text for downstream NLP tasks.
- Stemming and lemmatization reduce variations of words.
- POS tagging provides grammatical information.
- N-grams preserve local word-order patterns.
- One-hot encoding, BoW and TF-IDF convert text into numerical features.
- NER extracts useful real-world entities from unstructured text.